# Fit Details -> DataFrames

Quick loader for `fit_details.json` files with comparative plots.

Outputs:
- `rows_df`: one row per batch fit result
- `channels_df`: one row per `(batch row, channel)` result


In [1]:
from __future__ import annotations

import json
import re
from pathlib import Path
from typing import Dict, List, Tuple, Union

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 180)


In [ ]:
def _as_float(value):
    try:
        return float(value)
    except (TypeError, ValueError):
        return np.nan


def _expand_list(prefix: str, values) -> Dict[str, float]:
    out = {}
    if not isinstance(values, (list, tuple)):
        return out
    for i, val in enumerate(values, start=1):
        out[f"{prefix}_{i}"] = val
    return out


def _clean_key(key: str) -> str:
    cleaned = re.sub(r"[^0-9A-Za-z_]+", "_", str(key)).strip("_")
    return cleaned or "unknown"


def default_fit_files(search_root: Union[Path, str] = Path("..")) -> List[Path]:
    root = Path(search_root)
    candidates = sorted(
        {
            p.resolve()
            for p in root.rglob("fit_details.json")
            if ".git" not in p.parts
        }
    )
    return candidates


def load_fit_details(path: Union[Path, str]) -> Tuple[pd.DataFrame, pd.DataFrame, dict]:
    path = Path(path)
    with path.open("r", encoding="utf-8") as handle:
        payload = json.load(handle)

    dataset = path.parent.name
    param_defs = payload.get("parameters") if isinstance(payload, dict) else []

    param_keys = []
    if isinstance(param_defs, list):
        for i, p in enumerate(param_defs, start=1):
            if isinstance(p, dict):
                raw_key = p.get("key") or p.get("symbol") or p.get("name")
            else:
                raw_key = None
            param_keys.append(_clean_key(raw_key) if raw_key else f"p{i}")

    rows_out = []
    channels_out = []

    batch_rows = payload.get("batch_results") if isinstance(payload, dict) else []
    if not isinstance(batch_rows, list):
        batch_rows = []

    for row_index, entry in enumerate(batch_rows):
        if not isinstance(entry, dict):
            continue

        captures = entry.get("captures") if isinstance(entry.get("captures"), dict) else {}
        fit_results = entry.get("fit_results") if isinstance(entry.get("fit_results"), dict) else {}

        row = {
            "dataset": dataset,
            "source_path": str(path),
            "row_index": row_index,
            "file": entry.get("file"),
            "file_stem": entry.get("file_stem"),
            "y_channel": entry.get("y_channel"),
            "capture_id": captures.get("id"),
            "capture_freq": _as_float(captures.get("freq")),
            "fit_r2": fit_results.get("r2"),
            "fit_error": fit_results.get("error"),
        }

        row.update(_expand_list("fit_boundary_ratio", fit_results.get("boundary_ratios")))
        row.update(_expand_list("fit_boundary_value", fit_results.get("boundary_values")))

        params = fit_results.get("params")
        if isinstance(params, (list, tuple)):
            for i, value in enumerate(params, start=1):
                p_key = param_keys[i - 1] if i - 1 < len(param_keys) else f"p{i}"
                row[f"param_{p_key}"] = value

        rows_out.append(row)

        channel_results = fit_results.get("channel_results")
        if isinstance(channel_results, dict):
            for channel_name, channel_data in channel_results.items():
                if not isinstance(channel_data, dict):
                    continue
                ch_row = {
                    "dataset": dataset,
                    "source_path": str(path),
                    "row_index": row_index,
                    "file_stem": entry.get("file_stem"),
                    "capture_freq": _as_float(captures.get("freq")),
                    "channel": channel_name,
                    "channel_r2": channel_data.get("r2"),
                }
                ch_row.update(_expand_list("channel_boundary_ratio", channel_data.get("boundary_ratios")))
                ch_row.update(_expand_list("channel_boundary", channel_data.get("boundaries")))
                channels_out.append(ch_row)

    return pd.DataFrame(rows_out), pd.DataFrame(channels_out), payload


def load_many_fit_details(paths) -> Tuple[pd.DataFrame, pd.DataFrame]:
    row_frames = []
    channel_frames = []
    for p in paths:
        rows_df, channels_df, _ = load_fit_details(p)
        row_frames.append(rows_df)
        channel_frames.append(channels_df)

    rows = pd.concat(row_frames, ignore_index=True) if row_frames else pd.DataFrame()
    channels = pd.concat(channel_frames, ignore_index=True) if channel_frames else pd.DataFrame()
    return rows, channels


: 

In [ ]:
fit_files = default_fit_files()
print(f"Found {len(fit_files)} fit_details file(s):")
for path in fit_files:
    print(" -", path)

rows_df, channels_df = load_many_fit_details(fit_files)
print("\nrows_df shape:", rows_df.shape)
print("channels_df shape:", channels_df.shape)
rows_df.head()


: 

In [ ]:
param_cols = [c for c in rows_df.columns if c.startswith("param_")]
rows_df[["dataset", "file_stem", "capture_freq", "fit_r2"] + param_cols[:4]].head(10)


: 

In [ ]:
if rows_df.empty:
    print("No row-level fit data found.")
else:
    fig, ax = plt.subplots(figsize=(10, 4))
    for dataset, grp in rows_df.sort_values("capture_freq").groupby("dataset"):
        ax.plot(grp["capture_freq"], grp["fit_r2"], marker="o", markersize=3, linewidth=1.2, label=dataset)
    ax.set_title("Fit R2 vs Capture Frequency")
    ax.set_xlabel("Capture Frequency")
    ax.set_ylabel("Fit R2")
    ax.grid(alpha=0.3)
    ax.legend()
    plt.show()


: 

In [ ]:
if channels_df.empty:
    print("No per-channel fit data found.")
else:
    fig, ax = plt.subplots(figsize=(11, 5))
    grouped = channels_df.sort_values("capture_freq").groupby(["dataset", "channel"])
    for (dataset, channel), grp in grouped:
        ax.plot(
            grp["capture_freq"],
            grp["channel_r2"],
            marker=".",
            markersize=3,
            linewidth=1.0,
            label=f"{dataset} {channel}",
        )
    ax.set_title("Per-Channel R2 Comparison")
    ax.set_xlabel("Capture Frequency")
    ax.set_ylabel("Channel R2")
    ax.grid(alpha=0.3)
    ax.legend(ncol=2, fontsize=8)
    plt.show()


: 

In [ ]:
param_cols = [c for c in rows_df.columns if c.startswith("param_")]
if rows_df.empty or not param_cols:
    print("No parameter columns available for plotting.")
else:
    target_param = "param_f_mod" if "param_f_mod" in param_cols else param_cols[0]
    fig, ax = plt.subplots(figsize=(10, 4))
    for dataset, grp in rows_df.sort_values("capture_freq").groupby("dataset"):
        ax.plot(grp["capture_freq"], grp[target_param], marker="o", markersize=3, linewidth=1.2, label=dataset)
    ax.set_title(f"{target_param} vs Capture Frequency")
    ax.set_xlabel("Capture Frequency")
    ax.set_ylabel(target_param)
    ax.grid(alpha=0.3)
    ax.legend()
    plt.show()


: 

In [ ]:
if rows_df.empty:
    print("No data to summarize.")
else:
    summary_cols = ["fit_r2"] + [c for c in rows_df.columns if c.startswith("param_")][:3]
    summary = rows_df.groupby("dataset")[summary_cols].agg(["mean", "std", "min", "max"])
    summary.round(6)


: 